In [ ]:
import pandas as pd
import numpy as np
import spacy
import re
import math

In [ ]:
df = pd.read_excel('/content/llm.xlsx')

In [ ]:
df.head()

,text_before,text_after
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ..."
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...


In [ ]:
def count_syllables_ru(word):
    vowels = "аеёиоуыэюя"
    count = 0
    word = word.lower()
    for char in word:
        if char in vowels:
            count += 1
    return max(1, count)

def fres_ru(text):
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    num_sentences = len(sentences)

    words = re.findall(r'\b\w+\b', text.lower())
    num_words = len(words)

    num_syllables = sum(count_syllables_ru(word) for word in words)

    if num_words == 0 or num_sentences == 0:
        return 0.0

    words_per_sentence = num_words / num_sentences
    syllables_per_word = num_syllables / num_words

    fres = 206.835 - 1.3 * words_per_sentence - 60.1 * syllables_per_word
    return fres

In [ ]:
df['FRESru_before'] = df['text_before'].apply(fres_ru)
df['FRESru_after'] = df['text_after'].apply(fres_ru)

display(df.head())

,text_before,text_after,FRESru_before,FRESru_after
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...,-13.651667,-46.613361
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...,46.923000,48.895870
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...,-31.169237,-14.148766
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ...",-53.906176,-5.737109
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...,-19.523929,-46.116266


In [ ]:
def fkgl_ru(text):
    """Calculates Flesch-Kincaid Grade Level score for Russian text."""
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    num_sentences = len(sentences)

    words = re.findall(r'\b\w+\b', text.lower())
    num_words = len(words)

    num_syllables = sum(count_syllables_ru(word) for word in words)

    if num_words == 0 or num_sentences == 0:
        return 0.0  # Handle cases with no words or sentences

    asl = num_words / num_sentences  # Average Sentence Length
    asw = num_syllables / num_words  # Average Syllables per Word

    fkgl = 0.5 * asl + 8.4 * asw - 15.59
    return fkgl

df['FKGLru_before'] = df['text_before'].apply(fkgl_ru)
df['FKGLru_after'] = df['text_after'].apply(fkgl_ru)

display(df.head())

,text_before,text_after,FRESru_before,FRESru_after,FKGLru_before,FKGLru_after
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...,-13.651667,-46.613361,22.123333,29.541967
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...,46.923000,48.895870,14.718000,13.805652
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...,-31.169237,-14.148766,27.065085,27.550909
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ...",-53.906176,-5.737109,37.086471,19.319524
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...,-19.523929,-46.116266,24.960000,32.337215


In [ ]:
def smog_ru(text):
    """Calculates SMOG readability index for Russian text."""
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    num_sentences = len(sentences)

    words = re.findall(r'\b\w+\b', text.lower())
    num_words = len(words)

    multi_syllable_words_count = sum(1 for word in words if count_syllables_ru(word) >= 4)

    if num_sentences == 0:
        return 0.0

    sentences_in_tens = num_sentences / 10

    if num_sentences == 0:
        return 0.0
    try:
        smog = 1.1 * np.sqrt(multi_syllable_words_count * (64.6 / num_sentences) + 0.05)
        return smog
    except ZeroDivisionError:
        return 0.0
    except ValueError:
        return 0.0


df['SMOGru_before'] = df['text_before'].apply(smog_ru)
df['SMOGru_after'] = df['text_after'].apply(smog_ru)

display(df.head())

,text_before,text_after,FRESru_before,FRESru_after,FKGLru_before,FKGLru_after,SMOGru_before,SMOGru_after
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...,-13.651667,-46.613361,22.123333,29.541967,27.011291,35.365470
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...,46.923000,48.895870,14.718000,13.805652,19.770951,19.770951
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...,-31.169237,-14.148766,27.065085,27.550909,32.485404,33.081483
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ...",-53.906176,-5.737109,37.086471,19.319524,40.516003,22.829086
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...,-19.523929,-46.116266,24.960000,32.337215,28.649668,36.986018


In [ ]:
def count_characters_digits(text):
    """Counts letters and digits in a string."""
    return sum(c.isalnum() for c in text)

def count_words(text):
    """Counts words in a string."""
    words = re.findall(r'\b\w+\b', text)
    return len(words)

def count_sentences(text):
    """Counts sentences in a string."""
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return len(sentences)

def ariru(text):
    """Calculates Automated Readability Index (ARI) for Russian text."""
    num_chars_digits = count_characters_digits(text)
    num_words = count_words(text)
    num_sentences = count_sentences(text)

    if num_words == 0 or num_sentences == 0:
        return 0.0 # Handle cases with no words or sentences

    ari = 6.26 * (num_chars_digits / num_words) + 0.2805 * (num_words / num_sentences) - 31.04
    return ari

def cliru(text):
    """Calculates Coleman-Liau Index (CLI) for Russian text."""
    num_chars_digits = count_characters_digits(text)
    num_words = count_words(text)
    num_sentences = count_sentences(text)

    if num_words == 0:
        return 0.0 # Handle cases with no words

    L = (num_chars_digits / num_words) * 100 # Average number of characters per 100 words
    S = (num_sentences / num_words) * 100 # Average number of sentences per 100 words

    cli = 0.055 * L - 0.35 * S - 20.33
    return cli

df['ARIru_before'] = df['text_before'].apply(ariru)
df['ARIru_after'] = df['text_after'].apply(ariru)
df['CLIru_before'] = df['text_before'].apply(cliru)
df['CLIru_after'] = df['text_after'].apply(cliru)

display(df.head())

,text_before,text_after,FRESru_before,FRESru_after,FKGLru_before,FKGLru_after,SMOGru_before,SMOGru_after,ARIru_before,ARIru_after,CLIru_before,CLIru_after
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...,-13.651667,-46.613361,22.123333,29.541967,27.011291,35.365470,19.627962,26.876889,17.231538,21.891311
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...,46.923000,48.895870,14.718000,13.805652,19.770951,19.770951,5.269300,4.261935,4.010000,3.496087
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...,-31.169237,-14.148766,27.065085,27.550909,32.485404,33.081483,25.617123,20.083406,20.992034,14.189481
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ...",-53.906176,-5.737109,37.086471,19.319524,40.516003,22.829086,31.136088,19.788847,21.042549,18.159796
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...,-19.523929,-46.116266,24.960000,32.337215,28.649668,36.986018,21.528286,26.157725,17.705714,19.302911


In [ ]:
def ti_ru(text):
    """Calculates the TI readability index for Russian text."""
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    num_sentences = len(sentences)

    words = re.findall(r'\b\w+\b', text.lower())
    num_words = len(words)

    num_syllables = sum(count_syllables_ru(word) for word in words)

    if num_words == 0 or num_sentences == 0:
        return 0.0 # Handle cases with no words or sentences

    asl = num_words / num_sentences  # Average Sentence Length
    asw = num_syllables / num_words  # Average Syllables per Word

    if asl <= 1:
        return 0.0
    ti = asw * math.log10(asl)
    return ti

def lix_ru(text):
    """Calculates the LIX readability index for Russian text."""
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    num_sentences = len(sentences)

    words = re.findall(r'\b\w+\b', text) # Keep original case to count letters
    num_words = len(words)

    # Count words longer than 6 letters
    long_words_count = sum(1 for word in words if len(word) > 6)

    if num_words == 0 or num_sentences == 0:
        return 0.0 # Handle cases with no words or sentences

    # LIX(d) = A/B + 100 * C/A
    # A = number of words, B = number of sentences, C = number of words longer than 6 letters
    lix = (num_words / num_sentences) + 100 * (long_words_count / num_words)
    return lix


df['TIru_before'] = df['text_before'].apply(ti_ru)
df['TIru_after'] = df['text_after'].apply(ti_ru)
df['LIXru_before'] = df['text_before'].apply(lix_ru)
df['LIXru_after'] = df['text_after'].apply(lix_ru)

display(df.head())

,text_before,text_after,FRESru_before,FRESru_after,FKGLru_before,FKGLru_after,SMOGru_before,SMOGru_after,ARIru_before,ARIru_after,CLIru_before,CLIru_after,TIru_before,TIru_after,LIXru_before,LIXru_after
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...,-13.651667,-46.613361,22.123333,29.541967,27.011291,35.365470,19.627962,26.876889,17.231538,21.891311,4.274535,5.280214,77.051282,94.434426
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...,46.923000,48.895870,14.718000,13.805652,19.770951,19.770951,5.269300,4.261935,4.010000,3.496087,2.963633,2.901072,49.000000,49.086957
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...,-31.169237,-14.148766,27.065085,27.550909,32.485404,33.081483,25.617123,20.083406,20.992034,14.189481,4.882799,4.509297,88.822034,83.954545
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ...",-53.906176,-5.737109,37.086471,19.319524,40.516003,22.829086,31.136088,19.788847,21.042549,18.159796,5.524492,3.862034,113.745098,69.394558
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...,-19.523929,-46.116266,24.960000,32.337215,28.649668,36.986018,21.528286,26.157725,17.705714,19.302911,4.574053,5.355674,90.500000,100.259494


In [ ]:
import spacy
import re
import math

# Download the spaCy model for Russian
try:
    nlp = spacy.load('ru_core_news_sm')
except OSError:
    print("Downloading spaCy model 'ru_core_news_sm'...")
    from spacy.cli import download
    download('ru_core_news_sm')
    nlp = spacy.load('ru_core_news_sm')

In [ ]:
def calculate_dd_mdd(text):
    # Удаление символов '●'
    cleaned_text = text.replace('●', '')
    doc = nlp(cleaned_text)
    dd_list = []
    for token in doc:
        # пропускаем корень предложения (у него head == self)
        if token.head.i != token.i:
            dd = abs(token.head.i - token.i)
            dd_list.append(dd)
    mdd = sum(dd_list) / len(dd_list) if dd_list else 0
    total_dd = sum(dd_list) if dd_list else 0
    return dd_list, mdd, total_dd

def add_dependency_metrics(df, text_col):
    dd_results = []
    mdd_results = []
    tdd_results = []
    for text in df[text_col]:
        dd, mdd, tdd = calculate_dd_mdd(text)
        dd_results.append(dd)
        mdd_results.append(mdd)
        tdd_results.append(tdd)
    df['DD'+text_col] = dd_results
    df['MDD'+text_col] = mdd_results
    df['TotalDD'+text_col] = tdd_results
    return df

df = add_dependency_metrics(df, 'text_before')
df = add_dependency_metrics(df, 'text_after')

In [ ]:
df

,text_before,text_after,FRESru_before,FRESru_after,FKGLru_before,FKGLru_after,SMOGru_before,SMOGru_after,ARIru_before,ARIru_after,...,TIru_before,TIru_after,LIXru_before,LIXru_after,DDtext_before,MDDtext_before,TotalDDtext_before,DDtext_after,MDDtext_after,TotalDDtext_after
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...,-13.651667,-46.613361,22.123333,29.541967,27.011291,35.365470,19.627962,26.876889,...,4.274535,5.280214,77.051282,94.434426,"[1, 1, 1, 2, 1, 2, 1, 3, 1, 2, 2, 1, 11, 2, 1,...",4.162162,308,"[1, 1, 2, 2, 1, 3, 1, 2, 2, 1, 10, 2, 1, 3, 2,...",4.123457,334
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...,46.923000,48.895870,14.718000,13.805652,19.770951,19.770951,5.269300,4.261935,...,2.963633,2.901072,49.000000,49.086957,"[1, 11, 2, 1, 3, 1, 5, 1, 2, 1, 1, 5, 1, 1, 2,...",3.000000,84,"[1, 11, 2, 1, 3, 1, 5, 1, 2, 1, 4, 5, 3, 1, 1,...",2.518519,68
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...,-31.169237,-14.148766,27.065085,27.550909,32.485404,33.081483,25.617123,20.083406,...,4.882799,4.509297,88.822034,83.954545,"[3, 1, 1, 8, 1, 2, 1, 2, 3, 1, 1, 1, 1, 1, 2, ...",3.884058,268,"[1, 11, 1, 1, 1, 2, 1, 6, 1, 1, 2, 1, 1, 2, 3,...",4.542553,427
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ...",-53.906176,-5.737109,37.086471,19.319524,40.516003,22.829086,31.136088,19.788847,...,5.524492,3.862034,113.745098,69.394558,"[1, 15, 14, 1, 1, 2, 1, 2, 1, 6, 1, 3, 2, 1, 4...",4.081967,249,"[1, 2, 1, 2, 1, 3, 5, 4, 1, 2, 1, 6, 1, 2, 1, ...",2.833333,170
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...,-19.523929,-46.116266,24.960000,32.337215,28.649668,36.986018,21.528286,26.157725,...,4.574053,5.355674,90.500000,100.259494,"[8, 1, 1, 1, 1, 2, 1, 4, 1, 1, 2, 1, 4, 3, 1, ...",2.868852,175,"[7, 1, 1, 1, 3, 1, 2, 2, 1, 3, 4, 1, 2, 1, 5, ...",3.511111,316
5,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО в ...,-45.947353,-67.870072,30.562941,27.685362,31.878182,28.421397,29.129353,34.500130,...,5.315133,5.026068,101.647059,100.115942,"[9, 1, 1, 1, 1, 2, 2, 1, 6, 1, 2, 1, 2, 4, 1, ...",3.384615,132,"[8, 1, 1, 1, 4, 2, 1, 1, 1, 1, 2, 1, 5, 1, 2, ...",3.065574,187
6,"Прибор, установку или оборудование, подготовле...",Необходимо проверить перед включением: \n● при...,-45.841923,-73.092273,23.863846,30.537273,23.392787,31.878182,25.575731,31.755545,...,4.370085,5.613768,74.538462,85.636364,"[12, 1, 2, 1, 4, 1, 2, 1, 1, 2, 3, 5, 1, 1, 2, 4]",2.687500,43,"[1, 1, 2, 2, 1, 3, 2, 1, 3, 1, 1, 2, 1, 4, 2, ...",1.964286,55
7,"Растворы щелочей, концентрированных кислот и а...",Приготавливать растворы разрешается только:\n●...,7.613125,22.795215,17.347500,13.421828,17.684018,13.507325,17.072375,17.899790,...,3.574731,2.879134,75.375000,78.075269,"[7, 1, 2, 1, 3, 1, 5, 1, 3, 2, 1, 4, 2, 1, 3, ...",2.970588,101,"[2, 1, 4, 3, 1, 1, 5, 1, 1, 2, 1, 5, 2, 1, 3, ...",2.585366,106
8,Легковоспламеняющиеся и горючие жидкости необх...,ЛВЖ и ГЖ должны храниться: \n● в лабораторных ...,-20.389000,11.575000,24.126000,21.250000,26.524602,23.392787,21.294900,15.978333,...,4.529326,3.840515,93.000000,73.333333,"[3, 1, 2, 1, 1, 1, 2, 3, 2, 1, 6, 1, 2, 1, 2, ...",2.777778,75,"[3, 1, 2, 1, 4, 1, 2, 1, 26, 1, 1, 3, 7, 1, 5,...",4.589744,179
9,Основным источником травм при работе со стекля...,"Основным травмирующим фактором, связанным с ис...",-1.279286,-21.828636,22.410000,26.873636,25.007769,27.959265,17.280429,22.795288,...,4.134737,4.693589,78.000000,93.606061,"[1, 11, 1, 1, 3, 2, 1, 3, 1, 2, 1, 4, 1, 2, 1,...",2.677419,83,"[2, 1, 12, 1, 2, 1, 2, 1, 2, 1, 2, 1, 4, 9, 3,...",3.075000,123


In [ ]:
readability_metrics = ['FRESru', 'FKGLru', 'SMOGru', 'ARIru', 'CLIru', 'TIru', 'LIXru', 'MDDtext', 'TotalDDtext']

for metric in readability_metrics:
    before_col = f'{metric}_before'
    after_col = f'{metric}_after'
    deviation_col = f'{metric}_deviation'

    # Calculate deviation, handling division by zero or near-zero values
    df[deviation_col] = df.apply(
        lambda row: (row[after_col] - row[before_col]) / abs(row[before_col]) if row[before_col] != 0 else None,
        axis=1
    )

display(df.head())

,text_before,text_after,FRESru_before,FRESru_after,FKGLru_before,FKGLru_after,SMOGru_before,SMOGru_after,ARIru_before,ARIru_after,...,TotalDDtext_after,FRESru_deviation,FKGLru_deviation,SMOGru_deviation,ARIru_deviation,CLIru_deviation,TIru_deviation,LIXru_deviation,MDDtext_deviation,TotalDDtext_deviation
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...,-13.651667,-46.613361,22.123333,29.541967,27.011291,35.365470,19.627962,26.876889,...,334,-2.414481,0.335331,0.309285,0.369316,0.270421,0.235272,0.225605,-0.009299,0.084416
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...,46.923000,48.895870,14.718000,13.805652,19.770951,19.770951,5.269300,4.261935,...,68,0.042045,-0.061989,0.000000,-0.191176,-0.128158,-0.021109,0.001775,-0.160494,-0.190476
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...,-31.169237,-14.148766,27.065085,27.550909,32.485404,33.081483,25.617123,20.083406,...,427,0.546066,0.017950,0.018349,-0.216016,-0.324054,-0.076493,-0.054800,0.169538,0.593284
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ...",-53.906176,-5.737109,37.086471,19.319524,40.516003,22.829086,31.136088,19.788847,...,170,0.893572,-0.479068,-0.436542,-0.364440,-0.136996,-0.300925,-0.389912,-0.305890,-0.317269
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...,-19.523929,-46.116266,24.960000,32.337215,28.649668,36.986018,21.528286,26.157725,...,316,-1.362038,0.295562,0.290975,0.215040,0.090208,0.170881,0.107840,0.223873,0.805714


In [ ]:
df.to_excel('df.xlsx', index=False)

In [ ]:
#!pip install ruTS

In [ ]:
!pip install bert_score

In [ ]:
from bert_score import score

# Calculate BERTScore
# Setting lang='ru' for Russian language
P, R, F1 = score(df['text_after'].tolist(), df['text_before'].tolist(), lang='ru', verbose=True)

# Add the F1 scores to the DataFrame as it is a commonly used metric for BERTScore
df['BERTScore_F1'] = F1.tolist()

# You can also add Precision and Recall if needed
# df['BERTScore_P'] = P.tolist()
# df['BERTScore_R'] = R.tolist()

display(df.head())

calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 138.71 seconds, 0.37 sentences/sec


,text_before,text_after,FRESru_before,FRESru_after,FKGLru_before,FKGLru_after,SMOGru_before,SMOGru_after,ARIru_before,ARIru_after,...,FRESru_deviation,FKGLru_deviation,SMOGru_deviation,ARIru_deviation,CLIru_deviation,TIru_deviation,LIXru_deviation,MDDtext_deviation,TotalDDtext_deviation,BERTScore_F1
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...,-13.651667,-46.613361,22.123333,29.541967,27.011291,35.365470,19.627962,26.876889,...,-2.414481,0.335331,0.309285,0.369316,0.270421,0.235272,0.225605,-0.009299,0.084416,0.886535
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...,46.923000,48.895870,14.718000,13.805652,19.770951,19.770951,5.269300,4.261935,...,0.042045,-0.061989,0.000000,-0.191176,-0.128158,-0.021109,0.001775,-0.160494,-0.190476,0.904063
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...,-31.169237,-14.148766,27.065085,27.550909,32.485404,33.081483,25.617123,20.083406,...,0.546066,0.017950,0.018349,-0.216016,-0.324054,-0.076493,-0.054800,0.169538,0.593284,0.790718
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ...",-53.906176,-5.737109,37.086471,19.319524,40.516003,22.829086,31.136088,19.788847,...,0.893572,-0.479068,-0.436542,-0.364440,-0.136996,-0.300925,-0.389912,-0.305890,-0.317269,0.823961
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...,-19.523929,-46.116266,24.960000,32.337215,28.649668,36.986018,21.528286,26.157725,...,-1.362038,0.295562,0.290975,0.215040,0.090208,0.170881,0.107840,0.223873,0.805714,0.794504


In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spacy

# Загрузка русского языка для spaCy
try:
    nlp = spacy.load("ru_core_news_sm")
except OSError:
    print("Скачивание модели 'ru_core_news_sm' для spaCy...")
    from spacy.cli import download
    download("ru_core_news_sm")
    nlp = spacy.load("ru_core_news_sm")

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# Метрики Семантической Сохранности

def calculate_semantic_similarity(text1, text2):
    """
    Вычисляет косинусное сходство между двумя текстами, используя TF-IDF.
    """
    if not text1 or not text2:
        return 0.0

    vectorizer = TfidfVectorizer()
    try:
        # Создаем матрицу признаков для обоих текстов
        tfidf_matrix = vectorizer.fit_transform([text1, text2])
        # Вычисляем косинусное сходство
        similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return similarity
    except ValueError:
        # Если один из текстов пустой или содержит только стоп-слова,
        # TF-IDF может вызвать ошибку. В этом случае считаем сходство 0.
        return 0.0

df['semantic_similarity'] = df.apply(
    lambda row: calculate_semantic_similarity(row['text_before'], row['text_after']),
    axis=1
)

# Метрики Упрощения

stop_words = set(stopwords.words('russian'))

def calculate_simplification_metrics(text_before, text_after):
    """
    Вычисляет метрики упрощения: среднюю длину слова, среднюю длину предложения,
    и долю редких слов.
    """
    metrics = {}

    # 1. Средняя длина слова
    words_before = nltk.word_tokenize(text_before.lower())
    words_after = nltk.word_tokenize(text_after.lower())

    if words_before:
        metrics['avg_word_length_before'] = sum(len(word) for word in words_before) / len(words_before)
    else:
        metrics['avg_word_length_before'] = 0

    if words_after:
        metrics['avg_word_length_after'] = sum(len(word) for word in words_after) / len(words_after)
    else:
        metrics['avg_word_length_after'] = 0

    # 2. Средняя длина предложения
    sentences_before = nltk.sent_tokenize(text_before)
    sentences_after = nltk.sent_tokenize(text_after)

    if sentences_before:
        metrics['avg_sentence_length_before'] = len(words_before) / len(sentences_before)
    else:
        metrics['avg_sentence_length_before'] = 0

    if sentences_after:
        metrics['avg_sentence_length_after'] = len(words_after) / len(sentences_after)
    else:
        metrics['avg_sentence_length_after'] = 0

    # 3. Доля редких слов (слова, не являющиеся стоп-словами)
    num_non_stopwords_before = sum(1 for word in words_before if word not in stop_words)
    num_non_stopwords_after = sum(1 for word in words_after if word not in stop_words)

    if words_before:
        metrics['non_stopword_ratio_before'] = num_non_stopwords_before / len(words_before)
    else:
        metrics['non_stopword_ratio_before'] = 0

    if words_after:
        metrics['non_stopword_ratio_after'] = num_non_stopwords_after / len(words_after)
    else:
        metrics['non_stopword_ratio_after'] = 0

    # 4. Упрощение синтаксиса (используя spaCy)
    # Считаем среднюю глубину синтаксического дерева (упрощенный показатель)
    # или количество зависимостей.
    doc_before = nlp(text_before)
    doc_after = nlp(text_after)

    metrics['dependency_count_before'] = len(list(doc_before.sents))
    metrics['dependency_count_after'] = len(list(doc_after.sents))

    subordinating_conjunctions_before = sum(1 for token in doc_before if token.dep_ == 'mark')
    subordinating_conjunctions_after = sum(1 for token in doc_after if token.dep_ == 'mark')

    metrics['subordinating_conjunctions_before'] = subordinating_conjunctions_before
    metrics['subordinating_conjunctions_after'] = subordinating_conjunctions_after

    return pd.Series(metrics)

# Применяем функцию к DataFrame
simplification_metrics = df.apply(
    lambda row: calculate_simplification_metrics(row['text_before'], row['text_after']),
    axis=1
)

# Объединяем результаты с основным DataFrame
df = pd.concat([df, simplification_metrics], axis=1)


In [ ]:
df

,text_before,text_after,FRESru_before,FRESru_after,FKGLru_before,FKGLru_after,SMOGru_before,SMOGru_after,ARIru_before,ARIru_after,...,avg_word_length_before,avg_word_length_after,avg_sentence_length_before,avg_sentence_length_after,non_stopword_ratio_before,non_stopword_ratio_after,dependency_count_before,dependency_count_after,subordinating_conjunctions_before,subordinating_conjunctions_after
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...,-13.651667,-46.613361,22.123333,29.541967,27.011291,35.365470,19.627962,26.876889,...,6.168831,6.121951,25.666667,41.000000,0.714286,0.768293,3.0,2.0,0.0,0.0
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...,46.923000,48.895870,14.718000,13.805652,19.770951,19.770951,5.269300,4.261935,...,4.807692,4.600000,26.000000,25.000000,0.846154,0.840000,1.0,1.0,0.0,0.0
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...,-31.169237,-14.148766,27.065085,27.550909,32.485404,33.081483,25.617123,20.083406,...,6.782609,5.670330,34.500000,91.000000,0.855072,0.758242,2.0,1.0,1.0,0.0
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ...",-53.906176,-5.737109,37.086471,19.319524,40.516003,22.829086,31.136088,19.788847,...,6.467742,5.968254,62.000000,21.000000,0.790323,0.809524,1.0,3.0,1.0,1.0
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...,-19.523929,-46.116266,24.960000,32.337215,28.649668,36.986018,21.528286,26.157725,...,6.460317,6.527473,31.500000,45.500000,0.809524,0.780220,2.0,2.0,1.0,1.0
5,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО в ...,-45.947353,-67.870072,30.562941,27.685362,31.878182,28.421397,29.129353,34.500130,...,7.025000,7.516129,40.000000,20.666667,0.875000,0.951613,1.0,3.0,0.0,0.0
6,"Прибор, установку или оборудование, подготовле...",Необходимо проверить перед включением: \n● при...,-45.841923,-73.092273,23.863846,30.537273,23.392787,31.878182,25.575731,31.755545,...,6.705882,7.103448,17.000000,29.000000,0.823529,0.862069,1.0,1.0,0.0,0.0
7,"Растворы щелочей, концентрированных кислот и а...",Приготавливать растворы разрешается только:\n●...,7.613125,22.795215,17.347500,13.421828,17.684018,13.507325,17.072375,17.899790,...,6.305556,5.690476,18.000000,14.000000,0.777778,0.857143,2.0,3.0,0.0,0.0
8,Легковоспламеняющиеся и горючие жидкости необх...,ЛВЖ и ГЖ должны храниться: \n● в лабораторных ...,-20.389000,11.575000,24.126000,21.250000,26.524602,23.392787,21.294900,15.978333,...,6.571429,4.875000,28.000000,40.000000,0.785714,0.825000,1.0,1.0,0.0,0.0
9,Основным источником травм при работе со стекля...,"Основным травмирующим фактором, связанным с ис...",-1.279286,-21.828636,22.410000,26.873636,25.007769,27.959265,17.280429,22.795288,...,5.781250,5.926829,32.000000,41.000000,0.812500,0.878049,1.0,1.0,0.0,0.0


In [ ]:
# Создайте список столбцов в нужном порядке
new_column_order = ['text_before', 'text_after', 'FRESru_before', 'FRESru_after', 'FRESru_deviation', 'FKGLru_before', 'FKGLru_after', 'FKGLru_deviation', 'SMOGru_before', 'SMOGru_after', 'SMOGru_deviation', 'ARIru_before', 'ARIru_after', 'ARIru_deviation', 'CLIru_before', 'CLIru_after', 'CLIru_deviation', 'TIru_before', 'TIru_after', 'TIru_deviation', 'LIXru_before', 'LIXru_after', 'LIXru_deviation', 'DDtext_before', 'MDDtext_before', 'DDtext_after', 'MDDtext_after', 'MDDtext_deviation','TotalDDtext_before','TotalDDtext_after','TotalDDtext_deviation','BERTScore_F1']
# Переиндексируйте DataFrame, используя новый порядок столбцов
df = df[new_column_order]

# Выведите первые несколько строк DataFrame с новым порядком столбцов
display(df.head(30))

,text_before,text_after,FRESru_before,FRESru_after,FRESru_deviation,FKGLru_before,FKGLru_after,FKGLru_deviation,SMOGru_before,SMOGru_after,...,LIXru_deviation,DDtext_before,MDDtext_before,DDtext_after,MDDtext_after,MDDtext_deviation,TotalDDtext_before,TotalDDtext_after,TotalDDtext_deviation,BERTScore_F1
0,"ТО подлежат МИ, включая изделия, которые наход...",ТО подлежат медицинские изделия: \n● находящие...,-13.651667,-46.613361,-2.414481,22.123333,29.541967,0.335331,27.011291,35.365470,...,0.225605,"[1, 1, 1, 2, 1, 2, 1, 3, 1, 2, 2, 1, 11, 2, 1,...",4.162162,"[1, 1, 2, 2, 1, 3, 1, 2, 2, 1, 10, 2, 1, 3, 2,...",4.123457,-0.009299,308,334,0.084416,0.886535
1,Испытательное давление для стальных трубопрово...,Испытательные давления для стальных трубопрово...,46.923000,48.895870,0.042045,14.718000,13.805652,-0.061989,19.770951,19.770951,...,0.001775,"[1, 11, 2, 1, 3, 1, 5, 1, 2, 1, 1, 5, 1, 1, 2,...",3.000000,"[1, 11, 2, 1, 3, 1, 5, 1, 2, 1, 4, 5, 3, 1, 1,...",2.518519,-0.160494,84,68,-0.190476,0.904063
2,Если техническое обслуживание выполняется собс...,В случае исполнения работ по ТО собственной сл...,-31.169237,-14.148766,0.546066,27.065085,27.550909,0.017950,32.485404,33.081483,...,-0.054800,"[3, 1, 1, 8, 1, 2, 1, 2, 3, 1, 1, 1, 1, 1, 2, ...",3.884058,"[1, 11, 1, 1, 1, 2, 1, 6, 1, 1, 2, 1, 1, 2, 3,...",4.542553,0.169538,268,427,0.593284,0.790718
3,"После ремонта средства измерений (МИ), который...","Должен быть проведен КТС (либо поверка, в случ...",-53.906176,-5.737109,0.893572,37.086471,19.319524,-0.479068,40.516003,22.829086,...,-0.389912,"[1, 15, 14, 1, 1, 2, 1, 2, 1, 6, 1, 3, 2, 1, 4...",4.081967,"[1, 2, 1, 2, 1, 3, 5, 4, 1, 2, 1, 6, 1, 2, 1, ...",2.833333,-0.305890,249,170,-0.317269,0.823961
4,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО ме...,-19.523929,-46.116266,-1.362038,24.960000,32.337215,0.295562,28.649668,36.986018,...,0.107840,"[8, 1, 1, 1, 1, 2, 1, 4, 1, 1, 2, 1, 4, 3, 1, ...",2.868852,"[7, 1, 1, 1, 3, 1, 2, 2, 1, 3, 4, 1, 2, 1, 5, ...",3.511111,0.223873,175,316,0.805714,0.794504
5,Контроль качества работы системы технического ...,Оценку качества функционирования системы ТО в ...,-45.947353,-67.870072,-0.477127,30.562941,27.685362,-0.094153,31.878182,28.421397,...,-0.015063,"[9, 1, 1, 1, 1, 2, 2, 1, 6, 1, 2, 1, 2, 4, 1, ...",3.384615,"[8, 1, 1, 1, 4, 2, 1, 1, 1, 1, 2, 1, 5, 1, 2, ...",3.065574,-0.094262,132,187,0.416667,0.779594
6,"Прибор, установку или оборудование, подготовле...",Необходимо проверить перед включением: \n● при...,-45.841923,-73.092273,-0.594442,23.863846,30.537273,0.279646,23.392787,31.878182,...,0.148888,"[12, 1, 2, 1, 4, 1, 2, 1, 1, 2, 3, 5, 1, 1, 2, 4]",2.687500,"[1, 1, 2, 2, 1, 3, 2, 1, 3, 1, 1, 2, 1, 4, 2, ...",1.964286,-0.269103,43,55,0.279070,0.790449
7,"Растворы щелочей, концентрированных кислот и а...",Приготавливать растворы разрешается только:\n●...,7.613125,22.795215,1.994199,17.347500,13.421828,-0.226296,17.684018,13.507325,...,0.035824,"[7, 1, 2, 1, 3, 1, 5, 1, 3, 2, 1, 4, 2, 1, 3, ...",2.970588,"[2, 1, 4, 3, 1, 1, 5, 1, 1, 2, 1, 5, 2, 1, 3, ...",2.585366,-0.129679,101,106,0.049505,0.807411
8,Легковоспламеняющиеся и горючие жидкости необх...,ЛВЖ и ГЖ должны храниться: \n● в лабораторных ...,-20.389000,11.575000,1.567708,24.126000,21.250000,-0.119207,26.524602,23.392787,...,-0.211470,"[3, 1, 2, 1, 1, 1, 2, 3, 2, 1, 6, 1, 2, 1, 2, ...",2.777778,"[3, 1, 2, 1, 4, 1, 2, 1, 26, 1, 1, 3, 7, 1, 5,...",4.589744,0.652308,75,179,1.386667,0.776166
9,Основным источником травм при работе со стекля...,"Основным травмирующим фактором, связанным с ис...",-1.279286,-21.828636,-16.063144,22.410000,26.873636,0.199181,25.007769,27.959265,...,0.200078,"[1, 11, 1, 1, 3, 2, 1, 3, 1, 2, 1, 4, 1, 2, 1,...",2.677419,"[2, 1, 12, 1, 2, 1, 2, 1, 2, 1, 2, 1, 4, 9, 3,...",3.075000,0.148494,83,123,0.481928,0.861308


In [ ]:
calculate_dd_mdd("Прибор, установку, оборудование, самостоятельно подготовленные обучающимся к работе, необходимо проверить перед включением.")

([12, 1, 2, 1, 4, 2, 1, 3, 1, 1, 2, 4, 1, 1, 2, 4], 2.625, 42)

In [ ]:
calculate_dd_mdd("Прибор, установку, оборудование: самостоятельно подготовленные обучающимся к работе необходимо проверить перед включением.")

([11, 1, 2, 1, 4, 2, 1, 7, 1, 1, 2, 1, 1, 2, 4], 2.7333333333333334, 41)

In [ ]:
df.to_excel('df.xlsx', index=False)